<img src="https://img.shields.io/badge/Redes%20Complexas-Semana%2005-blue?style=for-the-badge" alt="badge"> <img src="https://img.shields.io/badge/NetworkX-3.x-orange?style=for-the-badge" alt="badge"> <img src="https://img.shields.io/badge/Plotly-interativo-9085e9?style=for-the-badge" alt="badge">

# 🧅 Hubs (cont.) · Decomposição em núcleos (*k-core*)

**Semana 05** · Análise de Redes Complexas · UFRN / DCA · ivanovitch.silva@ufrn.br

> Uma cebola tem camadas. Uma rede também. A pergunta desta semana é simples de enunciar e surpreendentemente profunda de responder:
> **quem está no coração da rede, e quem está apenas na casca?**

Na aula de Hubs vimos que o **grau** diz quantas conexões um nó tem. Mas grau é uma medida *local*. Um nó pode ter 30 vizinhos e, ainda assim, estar na periferia, porque todos os seus vizinhos são folhas. A **decomposição em núcleos** corrige isso com uma ideia recursiva: você só é importante se seus vizinhos também forem.

## 🎯 O que você vai conseguir fazer ao final

1. Explicar a diferença entre **k-core** e **k-shell** sem olhar os slides.
2. Executar o algoritmo de "descascar" à mão em um grafo pequeno e prever a saída de `nx.core_number`.
3. Implementar o algoritmo **O(m)** de Batagelj e Zaversnik do zero e comparar com o NetworkX.
4. Usar o arsenal completo do NetworkX (`core_number`, `k_core`, `k_shell`, `k_crust`, `k_corona`, `onion_layers`, `k_truss`).
5. Ler a "assinatura em camadas" de uma rede real (SocioPatterns) e de uma rede de coautoria científica (OpenAlex).
6. Construir visualizações interativas guiadas por princípios de **Gestalt**.

## 🗺️ Mapa entre slides e notebook

| Slide | Tema | Onde no notebook |
|---|---|---|
| 2 | K-Core vs K-Shell (a cebola) | Seção 2 |
| 3 a 15 | Descascando o grafo a…u, passo a passo | Seções 3 e 4 |
| 16 a 18 | SocioPatterns, "quantos k-cores?", 17-core | Seção 7 |
| 19 | Further Analysis (PDF, CDF, univariada, bivariada por k-shell) | Seção 8 |
| (novo) | Algoritmo O(m) e benchmark | Seção 6 |
| (novo) | Rede de coautoria via OpenAlex | Seção 9 |
| (novo) | Desafios | Seção 10 |

## 🧠 Por que Gestalt?

Os princípios de Gestalt descrevem como o cérebro agrupa o que vê. Cada figura deste notebook foi desenhada para explorar um deles, e a caixa **"Gestalt em ação"** antes de cada gráfico diz qual. A ideia é que a *forma* do gráfico faça o conteúdo grudar na memória.

| Princípio | Como aparece aqui |
|---|---|
| **Figura e fundo** | o núcleo em destaque, o resto da rede em cinza |
| **Proximidade** | *layout* em anéis concêntricos (a cebola literal) |
| **Similaridade** | uma única rampa de cor codificando o número de núcleo |
| **Destino comum** | nós removidos juntos desaparecem juntos na animação |
| **Continuidade** | curvas de CDF por camada, uma linha por *shell* |
| **Fechamento** | contorno (casco convexo) ao redor do núcleo máximo |

# 1 · Preparação do ambiente

Além do NetworkX e do pandas, usaremos **Plotly** para gráficos interativos e **gdown** para baixar o dataset SocioPatterns do Drive da disciplina. No Colab a célula abaixo instala o que faltar.

In [2]:
import importlib, subprocess, sys

for pacote in ["plotly", "gdown"]:
    if importlib.util.find_spec(pacote) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pacote], check=False)

import os, time, math, random, json
import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

print("NetworkX", nx.__version__, "| pandas", pd.__version__)

NetworkX 3.6.1 | pandas 2.2.3


## 1.1 · Paleta e funções de desenho

Uma decisão de projeto que vale explicitar. Em todo o notebook:

* **magnitude** (número de núcleo, grau) usa **uma única matiz azul, do claro para o escuro**. Cores de arco-íris parecem bonitas, mas embaralham a ordem na cabeça de quem lê.
* **identidade** (rede A vs rede B, estratégia X vs Y) usa uma paleta categórica com ordem fixa.
* o que não é o assunto da figura fica em **cinza**. Isso é o princípio de figura e fundo aplicado à risca.

As funções abaixo são pequenas por intenção. Leia o código, elas vão reaparecer nos desafios.

In [3]:
# Paleta (validada para daltonismo em pares adjacentes)
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#9085e9", "#e34948"]
AZUIS = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#1c5cab", "#104281", "#0d366b"]
CINZA_NO, CINZA_ARESTA, CINZA_TEXTO = "#c3c2b7", "#d9d8d2", "#52514e"
DESTAQUE = "#e34948"     # reservado para "o que acabou de ser removido"
SUPERFICIE = "#fcfcfb"

def rampa_azul(valores, vmin=None, vmax=None):
    # mapeia uma lista de números para a rampa AZUIS (sequencial, uma matiz)
    v = np.asarray(list(valores), dtype=float)
    vmin = v.min() if vmin is None else vmin
    vmax = v.max() if vmax is None else vmax
    if vmax == vmin:
        return [AZUIS[-1]] * len(v)
    idx = np.round((v - vmin) / (vmax - vmin) * (len(AZUIS) - 1)).astype(int)
    return [AZUIS[i] for i in idx]

def traco_arestas(g, pos, cor=CINZA_ARESTA, largura=1.0, opacidade=0.7, nome="arestas"):
    xs, ys = [], []
    for u, v in g.edges():
        xs += [pos[u][0], pos[v][0], None]
        ys += [pos[u][1], pos[v][1], None]
    return go.Scatter(x=xs, y=ys, mode="lines", hoverinfo="skip", name=nome,
                      line=dict(color=cor, width=largura), opacity=opacidade, showlegend=False)

def traco_nos(g, pos, cores, tamanhos=14, hover=None, rotulos=True, nome="nós",
              opacidade=1.0, borda="#ffffff"):
    nos = list(g.nodes())
    return go.Scatter(
        x=[pos[n][0] for n in nos], y=[pos[n][1] for n in nos],
        mode="markers+text" if rotulos else "markers",
        text=[str(n) for n in nos] if rotulos else None,
        textposition="middle center",
        textfont=dict(color=[("#0b0b0b" if c in AZUIS[:3] or c == CINZA_NO else "white") for c in
                             (cores if isinstance(cores, list) else [cores] * len(nos))], size=11),
        hovertext=hover if hover is not None else [str(n) for n in nos],
        hoverinfo="text",
        marker=dict(color=cores, size=tamanhos, line=dict(color=borda, width=1.5),
                    opacity=opacidade),
        name=nome, showlegend=False)

def figura_rede(tracos, titulo="", altura=520, largura=None):
    fig = go.Figure(data=tracos)
    fig.update_layout(
        title=dict(text=titulo, x=0.02, font=dict(size=18, color="#0b0b0b")),
        height=altura, width=largura, plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE,
        margin=dict(l=10, r=10, t=50, b=10), hovermode="closest",
        xaxis=dict(visible=False), yaxis=dict(visible=False, scaleanchor="x", scaleratio=1))
    return fig

print("Funções de desenho carregadas.")

Funções de desenho carregadas.


# 2 · O grafo dos slides e a intuição da cebola

Vamos recriar exatamente o grafo com nós `a` até `u` usado nos slides 3 a 15. Ele é pequeno o bastante para descascar à mão e rico o bastante para ter três camadas.

In [4]:
g = nx.Graph()
g.add_edges_from([('a','d'), ('b','e'), ('c','e'), ('d','i'), ('e','i'),
                  ('f','h'), ('g','h'), ('h','i'), ('h','j'), ('i','l'), ('j','n'),
                  ('l','m'), ('l','n'), ('l','k'), ('l','o'), ('m','u'),
                  ('n','k'), ('n','o'), ('n','s'), ('o','k'), ('o','p'), ('o','s'),
                  ('p','k'), ('p','s'), ('k','s'), ('k','u'), ('q','s'), ('r','s'), ('s','u')])

print(g)
pos_g = nx.spring_layout(g, seed=7, k=0.9)
graus = dict(g.degree())
hover = [f"<b>{n}</b><br>grau = {graus[n]}" for n in g.nodes()]
fig = figura_rede([traco_arestas(g, pos_g), traco_nos(g, pos_g, rampa_azul(graus.values()), 22, hover)],
                  "Grafo dos slides · cor = grau (quanto mais escuro, maior)")
fig.show()

Graph with 20 nodes and 29 edges


## 2.1 · Definições (guarde estas três frases)

Seja $G=(V,E)$ um grafo simples e não direcionado.

* O **k-core** de $G$ é o **maior subgrafo** $H \subseteq G$ em que **todo** nó tem grau $\geq k$ *dentro de $H$*. É obtido removendo, repetidamente, todo nó com grau menor que $k$ até não sobrar nenhum.
* O **número de núcleo** (*core number*) de um nó $v$, $c(v)$, é o **maior** $k$ tal que $v$ pertence ao k-core.
* O **k-shell** é o conjunto dos nós com $c(v) = k$ exatamente. Ou seja, $k\text{-shell} = k\text{-core} \setminus (k+1)\text{-core}$.

A **degenerescência** do grafo é o maior $k$ para o qual o k-core não é vazio. Também chamamos esse k-core de **núcleo principal** (*main core*).

Note duas coisas que confundem muita gente:

1. Os núcleos são **aninhados**. $0\text{-core} \supseteq 1\text{-core} \supseteq 2\text{-core} \supseteq \dots$ Cada camada da cebola envolve a próxima.
2. **Grau alto não garante núcleo alto.** O nó `s` tem grau 7, mas veja o que acontece com ele. E `i` tem grau 4 e fica em qual camada?

> **⏸️ Pare e pense.** Antes de rodar a próxima célula, escreva no papel quais nós você acha que estão no 3-core. Depois compare.

In [5]:
core = nx.core_number(g)
tabela = (pd.DataFrame({"nó": list(core), "grau": [graus[n] for n in core], "core_number": list(core.values())})
            .sort_values(["core_number", "grau"], ascending=False).reset_index(drop=True))
tabela.T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
nó,s,k,l,n,o,p,i,h,u,j,m,e,d,a,b,c,f,g,q,r
grau,7,6,5,5,5,3,4,4,3,2,2,3,2,1,1,1,1,1,1,1
core_number,3,3,3,3,3,3,2,2,2,2,2,1,1,1,1,1,1,1,1,1


In [6]:
print("Camadas presentes (valores distintos de core number):", sorted(set(core.values())))
for k in sorted(set(core.values()), reverse=True):
    print(f"{k}-shell -> {sorted(nx.k_shell(g, k).nodes())}")

Camadas presentes (valores distintos de core number): [1, 2, 3]
3-shell -> ['k', 'l', 'n', 'o', 'p', 's']
2-shell -> ['h', 'i', 'j', 'm', 'u']
1-shell -> ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'q', 'r']


> 🧠 **Gestalt em ação · Proximidade + Similaridade.** O *layout* em anéis coloca cada k-shell em um círculo, do núcleo (centro) para a casca (borda). A cor repete a mesma informação em outro canal (uma matiz, do claro ao escuro). Dois canais redundantes fazem o padrão "grudar".

In [7]:
def layout_cebola(g, core):
    # anéis concêntricos: a shell mais alta no centro, a mais baixa na borda.
    # o raio de cada anel cresce com a raiz do tamanho do k-core, para que os nós tenham espaço
    ks = sorted(set(core.values()), reverse=True)
    pos, acumulado = {}, 0
    for i, k in enumerate(ks):
        anel = sorted(n for n in g.nodes() if core[n] == k)
        acumulado += len(anel)
        r = math.sqrt(acumulado) + 0.6
        for j, n in enumerate(anel):
            ang = 2 * math.pi * j / len(anel) + i * 0.37
            pos[n] = np.array([r * math.cos(ang), r * math.sin(ang)])
    return pos, ks

pos_cebola, ks = layout_cebola(g, core)
cores = rampa_azul([core[n] for n in g.nodes()])
hover = [f"<b>{n}</b><br>grau = {graus[n]}<br>core number = {core[n]}" for n in g.nodes()]

fig = figura_rede([traco_arestas(g, pos_cebola), traco_nos(g, pos_cebola, cores, 24, hover)],
                  "A cebola: anel externo = 1-shell · anel do meio = 2-shell · centro = 3-shell")
# anéis-guia (fechamento): círculos tracejados marcando cada camada
for i, k in enumerate(ks):
    r = np.hypot(*pos_cebola[[n for n in g.nodes() if core[n] == k][0]])
    fig.add_shape(type="circle", x0=-r, y0=-r, x1=r, y1=r, line=dict(color=CINZA_NO, dash="dot", width=1))
    fig.add_annotation(x=-r * 0.72, y=r * 0.72, text=f"{k}-shell", showarrow=False, bgcolor="rgba(252,252,251,0.85)",
                       font=dict(color=CINZA_TEXTO))
fig.show()

# 3 · Descascando a cebola à mão

Os slides 5 a 15 mostram o algoritmo mais ingênuo possível, e é exatamente por isso que ele é ótimo para aprender:

```
k = 0
enquanto houver nós:
    v = nó de menor grau atual
    se grau(v) > k: k = grau(v)        # subimos uma camada
    core_number[v] = k
    remova v (e atualize os graus dos vizinhos)
```

Repare no detalhe crucial: **o k só sobe, nunca desce**. Se, ao remover um nó, um vizinho cair para grau 1 dentro de uma fase k = 2, ele ainda recebe core number 2. Esse "efeito cascata" é o que a animação abaixo mostra.

In [8]:
def descascar(g):
    # Registra, passo a passo, a remoção do nó de menor grau (versão didática, O(n^2))
    h = g.copy()
    k, passos = 0, []
    while h.number_of_nodes():
        graus_atuais = dict(h.degree())
        v = min(graus_atuais, key=lambda x: (graus_atuais[x], str(x)))   # desempate: ordem alfabética
        k = max(k, graus_atuais[v])
        passos.append({"passo": len(passos) + 1, "removido": v, "grau_no_momento": graus_atuais[v],
                       "core_number": k, "restantes": sorted(h.nodes())})
        h.remove_node(v)
    return passos

passos = descascar(g)
core_manual = {p["removido"]: p["core_number"] for p in passos}
assert core_manual == nx.core_number(g), "algo deu errado no descascar"

pd.DataFrame(passos)[["passo", "removido", "grau_no_momento", "core_number"]].T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
passo,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
removido,a,b,c,d,e,f,g,q,r,h,i,j,m,u,l,n,k,o,p,s
grau_no_momento,1,1,1,1,1,1,1,1,1,2,1,1,2,2,3,3,3,2,1,0
core_number,1,1,1,1,1,1,1,1,1,2,2,2,2,2,3,3,3,3,3,3


> **⏸️ Pare e pense.** Olhe a linha `grau_no_momento` acima. Em algum passo o grau do nó removido é **menor** que o `core_number` que ele recebeu? Esse é o efeito cascata. Quais nós sofreram esse efeito?

> 🧠 **Gestalt em ação · Destino comum + Figura e fundo.** Na animação, o nó que está sendo removido pisca em vermelho; os já removidos somem; o que resta continua azul. Use o *slider* devagar e tente prever o próximo nó a cair **antes** de avançar.

In [9]:
def animar_descascar(g, passos, pos, titulo="Descascando a cebola"):
    nos = list(g.nodes())
    cor_k = {1: AZUIS[2], 2: AZUIS[4], 3: AZUIS[6], 4: AZUIS[7]}
    frames, steps = [], []

    def estado(i):
        # i = número de passos já aplicados
        removidos = {p["removido"] for p in passos[:i]}
        proximo = passos[i]["removido"] if i < len(passos) else None
        k_atual = passos[i]["core_number"] if i < len(passos) else passos[-1]["core_number"]
        h = g.subgraph([n for n in nos if n not in removidos])
        cores, opac, hover = [], [], []
        for n in nos:
            if n in removidos:
                cores.append(CINZA_NO); opac.append(0.0); hover.append("")
            elif n == proximo:
                cores.append(DESTAQUE); opac.append(1.0)
                hover.append(f"<b>{n}</b> será removido<br>grau atual = {h.degree(n)}<br>recebe core number {k_atual}")
            else:
                cores.append(cor_k.get(k_atual, AZUIS[-1])); opac.append(1.0)
                hover.append(f"<b>{n}</b><br>grau atual = {h.degree(n)}")
        return h, cores, opac, hover, k_atual

    for i in range(len(passos) + 1):
        h, cores, opac, hover, k_atual = estado(i)
        legenda = (f"passo {i} · fase k = {k_atual} · restam {h.number_of_nodes()} nós"
                   if i < len(passos) else "fim · todos os nós têm core number")
        frames.append(go.Frame(name=str(i),
                               data=[traco_arestas(h, pos, largura=1.2),
                                     traco_nos(g, pos, cores, 24, hover, opacidade=opac)],
                               layout=go.Layout(title=dict(text=f"{titulo} · {legenda}"))))
        steps.append(dict(method="animate", label=str(i),
                          args=[[str(i)], dict(mode="immediate", frame=dict(duration=0, redraw=True),
                                               transition=dict(duration=200))]))

    fig = figura_rede(frames[0].data, f"{titulo} · passo 0", altura=560)
    fig.frames = frames
    fig.update_layout(
        updatemenus=[dict(type="buttons", x=1.0, y=1.08, xanchor="right", yanchor="bottom", direction="left",
                          buttons=[dict(label="▶ tocar", method="animate",
                                        args=[None, dict(frame=dict(duration=700, redraw=True), fromcurrent=True)]),
                                   dict(label="⏸ pausar", method="animate",
                                        args=[[None], dict(mode="immediate", frame=dict(duration=0, redraw=False))])])],
        sliders=[dict(active=0, steps=steps, x=0.05, len=0.9, y=0.0,
                      currentvalue=dict(prefix="passo: ", font=dict(color=CINZA_TEXTO)))])
    return fig

animar_descascar(g, passos, pos_g).show()

# 4 · O arsenal do NetworkX

Os slides usam três funções. O NetworkX tem sete. Todas vivem em `networkx.algorithms.core` e **todas partem do mesmo dicionário** `core_number`.

| Função | Retorna | Definição | Quando usar |
|---|---|---|---|
| `core_number(G)` | `dict` nó → k | maior k tal que o nó está no k-core | quase sempre é o ponto de partida |
| `k_core(G, k)` | subgrafo | nós com core number **≥ k** | "quem sobrevive à camada k" |
| `k_shell(G, k)` | subgrafo | nós com core number **= k** | "quem está exatamente na camada k" |
| `k_crust(G, k)` | subgrafo | nós com core number **≤ k** | o complemento do (k+1)-core, a "crosta" |
| `k_corona(G, k)` | subgrafo | nós do k-core com **exatamente k** vizinhos no k-core | a fronteira interna da camada |
| `onion_layers(G)` | `dict` nó → camada | refinamento do k-shell na ordem de remoção | "quem caiu primeiro dentro da mesma shell" |
| `k_truss(G, k)` | subgrafo | arestas que participam de **≥ k−2 triângulos** no subgrafo | núcleos mais "coesos" (baseados em triângulos) |

Duas pegadinhas que aparecem em prova e em projeto:

* `k_core(G)` **sem k** devolve o núcleo principal (a degenerescência).
* Todas exigem grafo **sem laços** (*self-loops*). Se seu dataset tiver `(v, v)`, remova antes com `G.remove_edges_from(nx.selfloop_edges(G))`.
* Em grafos **direcionados**, o grau usado é entrada + saída. Em **multigrafos** as funções levantam exceção; converta com `nx.Graph(M)`.

In [10]:
print("núcleo principal (k_core sem k):", sorted(nx.k_core(g).nodes()))
print("degenerescência:", max(nx.core_number(g).values()))
print()
for k in [1, 2, 3]:
    print(f"k = {k}")
    print(f"   k_core  : {sorted(nx.k_core(g, k).nodes())}")
    print(f"   k_shell : {sorted(nx.k_shell(g, k).nodes())}")
    print(f"   k_crust : {sorted(nx.k_crust(g, k).nodes())}")
    print(f"   k_corona: {sorted(nx.k_corona(g, k).nodes())}")

núcleo principal (k_core sem k): ['k', 'l', 'n', 'o', 'p', 's']
degenerescência: 3

k = 1
   k_core  : ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 'u']
   k_shell : ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'q', 'r']
   k_crust : ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'q', 'r']
   k_corona: ['a', 'b', 'c', 'f', 'g', 'q', 'r']
k = 2
   k_core  : ['h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 's', 'u']
   k_shell : ['h', 'i', 'j', 'm', 'u']
   k_crust : ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'm', 'q', 'r', 'u']
   k_corona: ['h', 'i', 'j', 'm']
k = 3
   k_core  : ['k', 'l', 'n', 'o', 'p', 's']
   k_shell : ['k', 'l', 'n', 'o', 'p', 's']
   k_crust : ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 'u']
   k_corona: ['l', 'p']


### 4.1 · `onion_layers`: o k-shell tem ordem interna

Dentro da mesma shell, nem todos os nós caem ao mesmo tempo. As **camadas de cebola** (Hébert-Dufresne, Grochow e Allard, 2016) registram essa ordem. Compare abaixo: a 1-shell tem 9 nós, mas eles se dividem em várias camadas.

In [11]:
camadas = nx.onion_layers(g)
df_onion = pd.DataFrame({"nó": list(camadas), "k_shell": [core[n] for n in camadas],
                         "camada_cebola": list(camadas.values())}).sort_values(["camada_cebola", "nó"])
df_onion.groupby(["k_shell", "camada_cebola"])["nó"].apply(lambda s: " ".join(s)).to_frame("nós")

nós
k_shell camada_cebola               
1       1              a b c f g q r
        2                        d e
2       3                    h i j m
        4                          u
3       5                        l p
        6                    k n o s

### 4.2 · `k_truss`: e se pedirmos triângulos em vez de vizinhos?

O k-core exige **k vizinhos**. O k-truss exige que **cada aresta** esteja em pelo menos **k − 2 triângulos**. É uma noção mais rígida de coesão. Uma estrela com 100 pontas tem um 1-core enorme e nenhum 3-truss.

> **⏸️ Pare e pense.** No nosso grafo, o 3-core tem 6 nós. Você acha que o 3-truss (cada aresta em ≥ 1 triângulo) será maior, menor ou igual?

In [12]:
for k in [3, 4, 5]:
    t = nx.k_truss(g, k)
    print(f"{k}-truss: {t.number_of_nodes()} nós, {t.number_of_edges()} arestas -> {sorted(t.nodes())}")
print("3-core   :", sorted(nx.k_core(g, 3).nodes()))

3-truss: 7 nós, 14 arestas -> ['k', 'l', 'n', 'o', 'p', 's', 'u']
4-truss: 6 nós, 12 arestas -> ['k', 'l', 'n', 'o', 'p', 's']
5-truss: 0 nós, 0 arestas -> []
3-core   : ['k', 'l', 'n', 'o', 'p', 's']


# 5 · Cinco fatos que vale memorizar (e um erro clássico)

1. $c(v) \leq \text{grau}(v)$ sempre. O core number nunca passa do grau.
2. Todo nó de um grafo **conexo** tem $c(v) \geq 1$. O 1-core é a rede inteira sem os nós isolados.
3. Um **clique** de $n$ nós é um $(n-1)$-core.
4. Uma **árvore** tem degenerescência 1. Não importa o quão grande.
5. Se $H \subseteq G$, então $c_H(v) \leq c_G(v)$. Remover arestas nunca aumenta o core number.

**Erro clássico:** achar que k-shell é "os nós de grau k". Não é. Rode a célula abaixo.

In [13]:
por_grau_2 = sorted(n for n in g.nodes() if graus[n] == 2)
shell_2 = sorted(nx.k_shell(g, 2).nodes())
print("nós de grau 2  :", por_grau_2)
print("2-shell         :", shell_2)
print("estão em ambos  :", sorted(set(por_grau_2) & set(shell_2)))
print("só na 2-shell   :", sorted(set(shell_2) - set(por_grau_2)), "  <- grau maior que 2, mas núcleo 2")

nós de grau 2  : ['d', 'j', 'm']
2-shell         : ['h', 'i', 'j', 'm', 'u']
estão em ambos  : ['j', 'm']
só na 2-shell   : ['h', 'i', 'u']   <- grau maior que 2, mas núcleo 2


In [14]:
# Fatos 3 e 4 verificados em código
clique = nx.complete_graph(8)
arvore = nx.random_labeled_tree(500, seed=1) if hasattr(nx, "random_labeled_tree") else nx.random_tree(500, seed=1)
print("clique K8 -> degenerescência", max(nx.core_number(clique).values()))
print("árvore com 500 nós -> degenerescência", max(nx.core_number(arvore).values()))

clique K8 -> degenerescência 7
árvore com 500 nós -> degenerescência 1


# 6 · O algoritmo O(m) de Batagelj e Zaversnik

A versão que animamos na Seção 3 procura o nó de menor grau a cada passo, o que custa $O(n)$ por remoção e $O(n^2)$ no total. Para redes com milhões de nós isso é inviável.

Batagelj e Zaversnik (2003) observaram que os graus são inteiros entre 0 e $\Delta$ e propuseram um **ordenamento por baldes** (*bin sort*): mantenha os nós ordenados por grau em um vetor, e quando o grau de um vizinho cair, **troque-o de posição** com o primeiro nó do balde dele. Cada aresta é olhada uma vez. Complexidade $O(n + m)$.

É exatamente o que `nx.core_number` faz por baixo dos panos. Vamos implementar, verificar que dá o mesmo resultado e medir.

In [15]:
def core_number_bz(g):
    # Batagelj & Zaversnik (2003), O(n + m). Assume grafo simples sem laços.
    grau = dict(g.degree())
    n = len(grau)
    if n == 0:
        return {}
    grau_max = max(grau.values())

    # 1) contar quantos nós há em cada grau (baldes)
    balde = [0] * (grau_max + 1)
    for v in grau:
        balde[grau[v]] += 1

    # 2) transformar contagens em posições iniciais de cada balde no vetor
    inicio = 0
    for d in range(grau_max + 1):
        qtd = balde[d]
        balde[d] = inicio
        inicio += qtd

    # 3) preencher o vetor ordenado por grau
    posicao, vetor = {}, [None] * n
    for v in grau:
        posicao[v] = balde[grau[v]]
        vetor[posicao[v]] = v
        balde[grau[v]] += 1

    # 4) restaurar os inícios dos baldes
    for d in range(grau_max, 0, -1):
        balde[d] = balde[d - 1]
    balde[0] = 0

    # 5) varrer o vetor uma única vez
    for i in range(n):
        v = vetor[i]
        for u in g.neighbors(v):
            if grau[u] > grau[v]:
                du, pu = grau[u], posicao[u]
                pw = balde[du]
                w = vetor[pw]
                if u != w:                        # troca u com o primeiro do balde
                    posicao[u], posicao[w] = pw, pu
                    vetor[pu], vetor[pw] = w, u
                balde[du] += 1
                grau[u] -= 1
    return grau   # ao final, grau[v] == core number de v

assert core_number_bz(g) == nx.core_number(g)
print("✅ core_number_bz coincide com nx.core_number no grafo dos slides")

✅ core_number_bz coincide com nx.core_number no grafo dos slides


### 6.1 · Benchmark honesto

Três implementações, mesma resposta. Vamos medir em uma rede Barabási-Albert (livre de escala, como a maioria das redes reais) com tamanho crescente. A versão ingênua entra só até onde é razoável esperar.

> 🧠 **Gestalt em ação · Continuidade.** Três linhas em escala log-log. A inclinação diz a complexidade: inclinação 1 é linear, inclinação 2 é quadrática. Passe o mouse para ler os tempos.

In [16]:
def cronometrar(fn, *args, repeticoes=3):
    melhor = math.inf
    for _ in range(repeticoes):
        t0 = time.perf_counter(); fn(*args); melhor = min(melhor, time.perf_counter() - t0)
    return melhor

def core_number_ingenuo(g):
    return {p["removido"]: p["core_number"] for p in descascar(g)}

tamanhos = [250, 500, 1000, 2000, 4000, 8000, 16000]
linhas = []
for n in tamanhos:
    ba = nx.barabasi_albert_graph(n, 3, seed=42)
    assert core_number_bz(ba) == nx.core_number(ba)
    linhas.append({"n": n, "m": ba.number_of_edges(), "implementação": "nx.core_number (NetworkX)", "segundos": cronometrar(nx.core_number, ba)})
    linhas.append({"n": n, "m": ba.number_of_edges(), "implementação": "core_number_bz (nosso, O(m))", "segundos": cronometrar(core_number_bz, ba)})
    if n <= 2000:
        linhas.append({"n": n, "m": ba.number_of_edges(), "implementação": "descascar ingênuo (O(n²))", "segundos": cronometrar(core_number_ingenuo, ba, repeticoes=1)})

bench = pd.DataFrame(linhas)
fig = px.line(bench, x="n", y="segundos", color="implementação", markers=True, log_x=True, log_y=True,
              color_discrete_sequence=CAT, title="Tempo de execução vs tamanho da rede (BA, m = 3)")
fig.update_traces(line=dict(width=2), marker=dict(size=9, line=dict(color="white", width=1)))
fig.update_layout(plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE, height=460,
                  legend=dict(orientation="h", y=1.08, x=0), xaxis=dict(gridcolor="#e1e0d9"), yaxis=dict(gridcolor="#e1e0d9"))
fig.show()
bench.pivot(index="n", columns="implementação", values="segundos").round(4)

implementação,"core_number_bz (nosso, O(m))",descascar ingênuo (O(n²)),nx.core_number (NetworkX)
n,,,
250,0.0006,0.0310,0.0016
500,0.0014,0.0737,0.0024
1000,0.0021,0.3195,0.0033
2000,0.0066,1.3335,0.0075
4000,0.0115,NaN,0.0201
8000,0.0256,NaN,0.0554
16000,0.0562,NaN,0.0901


> **⏸️ Pare e pense.** Por que o NetworkX, mesmo usando o mesmo algoritmo, pode ser mais lento (ou mais rápido) que a nossa versão? Dica: olhe o código-fonte com `nx.core_number??` e repare no que ele valida antes de começar.

# 7 · Rede real: SocioPatterns INFECTIOUS

Mesmo dataset da aula de Hubs. Visitantes da exposição *INFECTIOUS: STAY AWAY* (Science Gallery, Dublin, 2009) usaram crachás RFID; uma aresta é um contato face a face de ao menos 20 segundos. Cada linha do arquivo é **um contato**; contatos repetidos entre as mesmas pessoas viram **peso** da aresta.

Colunas de `out.sociopatterns-infectious`: `origem  destino  peso  timestamp` (guarde o *timestamp*, ele vai ser usado em um desafio).

Se o download do Drive falhar, a célula cria uma rede de contatos **sintética** no mesmo formato, para que o restante do notebook rode. Os números dos slides (por exemplo, o 17-core) valem só para o dataset real.

In [17]:
ARQUIVO = "out.sociopatterns-infectious"
URL_DRIVE = "https://drive.google.com/uc?id=1REW9CIhtNtdmoO5FrBPeCK5U39p5OMS5"
DADOS_REAIS = os.path.exists(ARQUIVO)

if not DADOS_REAIS:
    try:
        import gdown
        gdown.download(URL_DRIVE, ARQUIVO, quiet=True)
        DADOS_REAIS = os.path.exists(ARQUIVO) and os.path.getsize(ARQUIVO) > 1000
    except Exception as e:
        print("download falhou:", type(e).__name__)

if not DADOS_REAIS:
    print("⚠️ Usando rede de contatos SINTÉTICA (mesmo formato do arquivo real).")
    rng = np.random.default_rng(2026)
    n_pessoas = 410
    # grupos de visita com tamanhos heterogêneos (famílias, turmas escolares, excursões)
    tamanhos = []
    while sum(tamanhos) < n_pessoas:
        tamanhos.append(int(rng.choice([1, 2, 3, 4, 5, 6, 8, 10, 14, 20, 30, 40],
                                       p=[.12, .18, .16, .13, .1, .08, .06, .05, .04, .04, .025, .015])))
    grupo = np.repeat(np.arange(len(tamanhos)), tamanhos)[:n_pessoas]
    chegada = {gid: rng.uniform(0, 7 * 3600) for gid in range(len(tamanhos))}   # grupos chegam ao longo de 7 h
    atividade = rng.lognormal(0, 1.1, n_pessoas)                                # visitantes mais ou menos sociáveis
    gs = nx.Graph(); gs.add_nodes_from(range(n_pessoas))
    for gid in range(len(tamanhos)):                                             # contatos dentro do grupo
        membros = np.where(grupo == gid)[0]
        p_in = 0.95 if len(membros) <= 6 else (0.7 if len(membros) < 20 else 0.85)
        for i in range(len(membros)):
            for j in range(i + 1, len(membros)):
                if rng.random() < p_in * min(1, (atividade[membros[i]] * atividade[membros[j]]) ** 0.5):
                    gs.add_edge(int(membros[i]), int(membros[j]))
    ordem = sorted(range(len(tamanhos)), key=lambda x: chegada[x])              # grupos de horário próximo se cruzam
    for a in range(n_pessoas):
        social = atividade[a] > np.quantile(atividade, 0.7)                     # 30% circulam entre grupos
        monitor = atividade[a] > np.quantile(atividade, 0.96)                   # "monitores" falam com todo mundo
        k_extra = rng.poisson(18.0) if monitor else (rng.poisson(4.0) if social else (1 if rng.random() < 0.1 else 0))
        for _ in range(k_extra):
            idx = ordem.index(grupo[a])
            gb = int(rng.integers(len(ordem))) if monitor else ordem[int(np.clip(idx + rng.integers(-2, 3), 0, len(ordem) - 1))]
            b = int(rng.choice(np.where(grupo == gb)[0]))
            if a != b:
                gs.add_edge(a, b)
    comps = sorted(nx.connected_components(gs), key=len, reverse=True)
    for comp in comps[1:]:                                                       # rede conexa, como a real
        gs.add_edge(int(next(iter(comp))), int(rng.choice(list(comps[0]))))
    with open(ARQUIVO, "w") as f:
        f.write("% sym unweighted\n% sintético\n")
        for u, v in gs.edges():                                                  # contatos repetidos, com timestamps
            t0 = max(chegada[grupo[u]], chegada[grupo[v]])
            for _ in range(1 + rng.geometric(0.15)):
                f.write(f"{u + 1} {v + 1} 1 {int(t0 + rng.uniform(0, 5400))}\n")

df = pd.read_csv(ARQUIVO, sep=r"\s+", comment="%", header=None, names=["p1", "p2", "peso", "timestamp"])
print("fonte:", "dataset real" if DADOS_REAIS else "sintético", "| contatos registrados:", len(df))
df.head()

fonte: dataset real | contatos registrados: 17298


,p1,p2,peso,timestamp
0,100,101,1,1247658439
1,100,101,1,1247660019
2,100,102,1,1247658579
3,100,103,1,1247658159
4,100,103,1,1247658179


### 7.1 · Do arquivo ao grafo (sem `iterrows`)

Na aula passada montamos o grafo com um `for` sobre `df.iterrows()`. Funciona, mas é lento e esconde a ideia. A ideia é: **agrupar contatos repetidos e contar**. O pandas faz isso em uma linha e o NetworkX lê o resultado direto.

In [18]:
# normaliza a ordem (p1 < p2) para que (3, 7) e (7, 3) sejam a mesma aresta
pares = pd.DataFrame({"u": df[["p1", "p2"]].min(axis=1), "v": df[["p1", "p2"]].max(axis=1)})
arestas = pares.groupby(["u", "v"]).size().reset_index(name="weight")
arestas = arestas[arestas.u != arestas.v]                      # remove laços, se houver

g2 = nx.from_pandas_edgelist(arestas, "u", "v", edge_attr="weight")
print(g2)
print("conexo?", nx.is_connected(g2), "| componentes:", nx.number_connected_components(g2))

Graph with 410 nodes and 2765 edges
conexo? True | componentes: 1


### 7.2 · Quantas camadas essa rede tem?

O slide 17 pergunta: *how many k-cores does this network have?* A resposta no dataset real é `{1, 2, …, 13, 17}`. Repare no **salto de 13 para 17**: não existe 14-, 15- nem 16-shell. Isso significa que, ao descascar a 13-shell, o que sobra já é um bloco tão denso que todo mundo tem pelo menos 17 vizinhos dentro dele.

> 🧠 **Gestalt em ação · Similaridade + Fechamento.** Barras com a mesma matiz azul, mais escuras quanto maior o k. O olho lê "uma sequência", e o buraco entre 13 e 17 salta à vista porque quebra a continuidade.

In [19]:
core2 = nx.core_number(g2)
grau2 = dict(g2.degree())
assert core_number_bz(g2) == core2

ks2 = sorted(set(core2.values()))
kmax = max(ks2)
print("camadas presentes:", ks2)
print(f"núcleo principal: {kmax}-core com {len(nx.k_core(g2, kmax))} nós")

tam_shell = pd.Series(core2).value_counts().sort_index()
tam_core = pd.Series({k: sum(v >= k for v in core2.values()) for k in range(0, kmax + 1)})

fig = make_subplots(rows=1, cols=2, subplot_titles=("tamanho de cada k-shell", "tamanho de cada k-core (acumulado)"))
fig.add_bar(x=tam_shell.index, y=tam_shell.values, marker_color=rampa_azul(tam_shell.index, 0, kmax),
            hovertemplate="%{x}-shell: %{y} nós<extra></extra>", row=1, col=1)
fig.add_bar(x=tam_core.index, y=tam_core.values, marker_color=rampa_azul(tam_core.index, 0, kmax),
            hovertemplate="%{x}-core: %{y} nós<extra></extra>", row=1, col=2)
fig.update_layout(showlegend=False, height=400, plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE,
                  title="Assinatura em camadas do SocioPatterns", bargap=0.15)
fig.update_xaxes(title_text="k", dtick=1)
fig.update_yaxes(gridcolor="#e1e0d9")
fig.show()

camadas presentes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 17]
núcleo principal: 17-core com 32 nós


### 7.3 · A cebola inteira

Mesmo *layout* concêntrico da Seção 2, agora com centenas de nós. Cada anel é uma k-shell; o centro é o núcleo principal. Passe o mouse para ver quem é quem.

> 🧠 **Gestalt em ação · Proximidade.** Nós na mesma camada estão no mesmo anel. As arestas cinza, propositalmente apagadas, mostram um padrão que costuma surpreender: quase todas as arestas de um nó da casca apontam **para dentro**.

In [20]:
pos_cebola2, ks_desc = layout_cebola(g2, core2)
hover2 = [f"<b>visitante {n}</b><br>grau = {grau2[n]}<br>core number = {core2[n]}" for n in g2.nodes()]
fig = figura_rede([traco_arestas(g2, pos_cebola2, largura=0.4, opacidade=0.35),
                   traco_nos(g2, pos_cebola2, rampa_azul([core2[n] for n in g2.nodes()], 1, kmax), 9, hover2, rotulos=False)],
                  f"SocioPatterns em anéis · centro = {kmax}-core · borda = 1-shell", altura=680)
for k in ks_desc:                       # rótulo só nos anéis com gente suficiente para não poluir
    anel = [n for n in g2.nodes() if core2[n] == k]
    if len(anel) >= 8 or k == kmax:
        r = np.hypot(*pos_cebola2[anel[0]])
        fig.add_annotation(x=0, y=r, text=f"{k}", showarrow=False, bgcolor="rgba(252,252,251,0.85)",
                           font=dict(color=CINZA_TEXTO, size=11))
fig.show()

### 7.4 · Figura e fundo: gire o k

O slide 18 destaca o 17-core (vermelho) e a 13-shell (azul) sobre a rede cinza. Aqui você escolhe o k no menu. Repare como o núcleo, **espacialmente**, é um bloco compacto no *layout* de molas, mesmo sem ninguém ter dito ao algoritmo de *layout* nada sobre núcleos. A densidade de arestas faz o trabalho.

> 🧠 **Gestalt em ação · Figura e fundo + Fechamento.** Tudo que não é o k-core escolhido vira cinza claro. O casco convexo (linha tracejada) "fecha" o núcleo em uma forma única, que o cérebro reconhece como um objeto.

In [21]:
from scipy.spatial import ConvexHull

pos_spring2 = nx.spring_layout(g2, seed=123456789, k=0.3)

def casco(nos, pos):
    pts = np.array([pos[n] for n in nos])
    if len(pts) < 3:
        return None
    h = ConvexHull(pts)
    ciclo = list(h.vertices) + [h.vertices[0]]
    return pts[ciclo, 0], pts[ciclo, 1]

nos2 = list(g2.nodes())
def cores_para_k(k):
    return [AZUIS[6] if core2[n] >= k else CINZA_NO for n in nos2]

fig = figura_rede([traco_arestas(g2, pos_spring2, largura=0.4, opacidade=0.4),
                   traco_nos(g2, pos_spring2, cores_para_k(kmax), 9, hover2, rotulos=False)],
                  altura=640)

opcoes_k = [k for k in ks2 if k >= 2]
botoes, formas_por_k = [], {}
for k in opcoes_k:
    hx = casco(nx.k_core(g2, k).nodes(), pos_spring2)
    formas = [dict(type="path", path="M " + " L ".join(f"{x},{y}" for x, y in zip(*hx)) + " Z",
                   line=dict(color=AZUIS[6], dash="dot", width=1.5), fillcolor="rgba(42,120,214,0.08)")] if hx else []
    botoes.append(dict(label=f"{k}-core ({len(nx.k_core(g2, k))} nós)", method="update",
                       args=[{"marker.color": [cores_para_k(k)]},
                             {"title.text": f"SocioPatterns · em destaque: {k}-core", "shapes": formas}]))
# estado inicial = núcleo principal
fig.update_layout(title_text=f"SocioPatterns · em destaque: {kmax}-core", shapes=botoes[-1]["args"][1]["shapes"],
                  updatemenus=[dict(type="dropdown", x=0.01, y=0.99, xanchor="left", yanchor="top",
                                    active=len(botoes) - 1, buttons=botoes, bgcolor="white")])
fig.show()

### 7.5 · Hub não é sinônimo de núcleo

Esta é a figura mais importante da semana. Cada ponto é um visitante. No eixo x, o grau (quantas pessoas ele encontrou). No eixo y, o número de núcleo. Se as duas medidas fossem "a mesma coisa", os pontos estariam em uma diagonal.

> 🧠 **Gestalt em ação · Similaridade.** Os pontos vermelhos são **hubs sem núcleo**: grau no top 10%, mas core number no máximo **metade** da degenerescência. Alguém que falou com muita gente, mas com gente que não fala entre si. Para uma epidemia, quem é mais perigoso, o hub ou o nó do núcleo? (guarde a pergunta para o Desafio 3)

In [22]:
tab2 = pd.DataFrame({"visitante": nos2, "grau": [grau2[n] for n in nos2], "core": [core2[n] for n in nos2]})
p90 = tab2.grau.quantile(0.90)
eh_hub = tab2.grau >= p90
sem_nucleo = eh_hub & (tab2.core <= kmax / 2)
if sem_nucleo.sum() == 0:      # se nenhum hub estiver tão abaixo, pega os 5 hubs mais distantes da diagonal
    sem_nucleo = tab2.index.isin(tab2[eh_hub].assign(gap=lambda d: d.grau - d.core).nlargest(5, "gap").index)
tab2["tipo"] = np.where(sem_nucleo, "hub sem núcleo", "demais")
print(f"hubs (grau >= {p90:.0f}): {int(eh_hub.sum())} | hubs sem núcleo (core <= {kmax / 2:.0f}): {int(tab2.tipo.eq('hub sem núcleo').sum())}")

rng = np.random.default_rng(0)
fig = go.Figure()
for tipo, cor, tam in [("demais", AZUIS[3], 8), ("hub sem núcleo", DESTAQUE, 12)]:
    sub = tab2[tab2.tipo == tipo]
    fig.add_scatter(x=sub.grau + rng.uniform(-0.3, 0.3, len(sub)), y=sub.core + rng.uniform(-0.25, 0.25, len(sub)),
                    mode="markers", name=tipo, marker=dict(color=cor, size=tam, opacity=0.8, line=dict(color="white", width=1)),
                    customdata=sub[["visitante", "grau", "core"]].values,
                    hovertemplate="visitante %{customdata[0]}<br>grau = %{customdata[1]}<br>core = %{customdata[2]}<extra></extra>")
fig.add_shape(type="line", x0=0, y0=0, x1=tab2.grau.max(), y1=tab2.grau.max(), line=dict(color=CINZA_NO, dash="dot"))
fig.add_annotation(x=tab2.grau.max() * 0.55, y=tab2.grau.max() * 0.55, text="core = grau (limite teórico)", showarrow=False,
                   textangle=-30, font=dict(color=CINZA_TEXTO))
fig.update_layout(title="Grau × número de núcleo · cada ponto é um visitante", xaxis_title="grau", yaxis_title="core number",
                  height=480, plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE, legend=dict(orientation="h", y=1.08, x=0),
                  yaxis=dict(range=[0, tab2.core.max() + 2], gridcolor="#e1e0d9"), xaxis=dict(gridcolor="#e1e0d9"))
fig.show()
print("correlação de Spearman grau × core:", round(tab2[["grau", "core"]].corr("spearman").iloc[0, 1], 3))
tab2[tab2.tipo == "hub sem núcleo"].sort_values("grau", ascending=False).head(8)

hubs (grau >= 25): 44 | hubs sem núcleo (core <= 8): 1


correlação de Spearman grau × core: 0.918


,visitante,grau,core,tipo
17,42,26,8,hub sem núcleo


# 8 · Further Analysis: centralidades por camada

O slide 19 pede: investigar PDF, CDF e análises univariada e bivariada das centralidades, **para cada k-shell** e para o núcleo principal. Vamos fazer isso de forma interativa.

Primeiro, uma tabela longa com quatro centralidades por nó. Para não afogar o gráfico em 14 camadas, agrupamos as shells em faixas (casca, meio, núcleo). Esse agrupamento é uma escolha de análise, e é justo discutir se é a melhor.

In [23]:
cent = pd.DataFrame({
    "visitante": nos2,
    "grau": pd.Series(nx.degree_centrality(g2)),
    "proximidade": pd.Series(nx.closeness_centrality(g2)),
    "intermediação": pd.Series(nx.betweenness_centrality(g2, seed=1)),
    "autovetor": pd.Series(nx.eigenvector_centrality(g2, max_iter=2000)),
}).reset_index(drop=True)
cent["core"] = cent.visitante.map(core2)

def faixa(k):
    if k == kmax: return f"núcleo ({kmax}-core)"
    if k <= 4:    return "casca (1 a 4)"
    return f"meio (5 a {kmax - 1})"
cent["faixa"] = cent.core.map(faixa)
ordem_faixas = ["casca (1 a 4)", f"meio (5 a {kmax - 1})", f"núcleo ({kmax}-core)"]
cent.groupby("faixa")[["grau", "proximidade", "intermediação", "autovetor"]].median().reindex(ordem_faixas).round(4)

,grau,proximidade,intermediação,autovetor
faixa,,,,
casca (1 a 4),0.0073,0.2348,0.0000,0.0003
meio (5 a 16),0.0342,0.2926,0.0027,0.0048
núcleo (17-core),0.0611,0.3024,0.0041,0.1659


### 8.1 · CDF empírica por faixa

> 🧠 **Gestalt em ação · Continuidade.** Uma curva por faixa, mesma matiz, do claro (casca) ao escuro (núcleo). Se as curvas se **separam**, a centralidade discrimina as camadas. Se se **cruzam ou sobrepõem**, não discrimina. Qual das quatro separa melhor?

In [24]:
longo = cent.melt(id_vars=["visitante", "core", "faixa"], value_vars=["grau", "proximidade", "intermediação", "autovetor"],
                  var_name="centralidade", value_name="valor")
cores_faixa = {ordem_faixas[0]: AZUIS[1], ordem_faixas[1]: AZUIS[4], ordem_faixas[2]: AZUIS[7]}
fig = px.ecdf(longo, x="valor", color="faixa", facet_col="centralidade", facet_col_wrap=2,
              category_orders={"faixa": ordem_faixas}, color_discrete_map=cores_faixa,
              title="CDF empírica de cada centralidade, por faixa de núcleo")
fig.update_xaxes(matches=None, showticklabels=True, gridcolor="#e1e0d9")
fig.update_yaxes(gridcolor="#e1e0d9")
fig.update_traces(line=dict(width=2.5))
fig.update_layout(height=640, plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE,
                  legend=dict(orientation="h", y=-0.12, x=0, title_text=""))
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.for_each_yaxis(lambda ax: ax.update(title_text="probabilidade") if ax.title.text else None)
fig.show()

### 8.2 · PDF por shell (univariada), com o k na mão

Agora sem agrupar. Uma distribuição do grau para **cada** k-shell, empilhadas como *violinos*. Escolha a centralidade no menu.

> 🧠 **Gestalt em ação · Similaridade.** Os violinos ficam mais escuros à medida que o k cresce. A pergunta a responder olhando o gráfico: a partir de que k o grau "descola" da casca?

In [25]:
metricas = ["grau", "proximidade", "intermediação", "autovetor"]
fig = go.Figure()
for i, m in enumerate(metricas):
    for k in ks2:
        sub = cent[cent.core == k]
        fig.add_violin(y=sub[m], x=[f"{k}" ] * len(sub), name=f"{k}-shell", visible=(i == 0), showlegend=False,
                       line_color=rampa_azul([k], 1, kmax)[0], fillcolor=rampa_azul([k], 1, kmax)[0], opacity=0.75,
                       points="all", jitter=0.35, pointpos=0, marker=dict(size=4, color=CINZA_TEXTO, opacity=0.5),
                       box_visible=True, meanline_visible=False, spanmode="hard",
                       customdata=sub.visitante, hovertemplate="visitante %{customdata}<br>" + m + " = %{y:.4f}<extra></extra>")
n_por_metrica = len(ks2)
botoes = [dict(label=m, method="update",
               args=[{"visible": [j // n_por_metrica == i for j in range(len(metricas) * n_por_metrica)]},
                     {"yaxis.title.text": m}]) for i, m in enumerate(metricas)]
fig.update_layout(title="Distribuição de cada centralidade por k-shell", xaxis_title="k-shell", yaxis_title="grau",
                  height=520, plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE, violingap=0.15,
                  updatemenus=[dict(type="dropdown", x=1.0, y=1.12, xanchor="right", buttons=botoes)],
                  yaxis=dict(gridcolor="#e1e0d9"))
fig.show()

### 8.3 · Bivariada: intermediação × grau, colorida pelo núcleo

Os nós de maior intermediação são os do núcleo? Nem sempre. Intermediação premia **pontes**, e pontes ficam entre blocos, não dentro deles.

> 🧠 **Gestalt em ação · Figura e fundo.** O núcleo principal ganha borda vermelha. Repare se ele ocupa o canto superior direito (grau alto, intermediação alta) ou se aparece um grupo de pontos cinza de intermediação alta fora do núcleo: são os "pontes".

In [26]:
cent["no_nucleo"] = cent.core == kmax
fig = go.Figure()
fig.add_scatter(x=cent.grau, y=cent["intermediação"], mode="markers",
                marker=dict(color=cent.core, colorscale=[[0, AZUIS[0]], [1, AZUIS[7]]], cmin=1, cmax=kmax, size=9,
                            colorbar=dict(title="core", thickness=12), opacity=0.85,
                            line=dict(color=np.where(cent.no_nucleo, DESTAQUE, "white"), width=np.where(cent.no_nucleo, 2.5, 1))),
                customdata=cent[["visitante", "core"]].values,
                hovertemplate="visitante %{customdata[0]}<br>core = %{customdata[1]}<br>grau = %{x:.3f}<br>intermediação = %{y:.4f}<extra></extra>")
fig.update_layout(title=f"Grau × intermediação · borda vermelha = {kmax}-core", xaxis_title="centralidade de grau",
                  yaxis_title="centralidade de intermediação", height=480, plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE,
                  xaxis=dict(gridcolor="#e1e0d9"), yaxis=dict(gridcolor="#e1e0d9"))
fig.show()

# 9 · Uma rede ousada: coautoria científica via OpenAlex

Contatos face a face são um tipo de rede. Vamos olhar outro, com uma pergunta provocadora: **qual é o núcleo da produção científica da UFRN?**

O [OpenAlex](https://openalex.org) é uma base aberta de publicações, sem chave de API. Vamos buscar trabalhos recentes com pelo menos um autor da UFRN (ROR `04wn09761`), montar a rede de **coautoria** (dois pesquisadores se conectam se assinaram um artigo juntos) e decompor em núcleos.

Um detalhe metodológico que muda tudo: artigos com **centenas de autores** (grandes colaborações em física, medicina, astronomia) criam cliques gigantes e um k-core artificial. Vamos limitar a 25 autores por trabalho e deixar a discussão desse corte para o Desafio 6.

Se a API não responder (rede bloqueada, limite de requisições), a célula gera uma rede de coautoria **sintética** com a mesma mecânica (autores com produtividade Zipf assinando artigos em grupos).

In [27]:
import requests

ROR_UFRN = "https://ror.org/04wn09761"
MAX_AUTORES = 25
PAGINAS = 5            # 5 páginas x 200 trabalhos = até 1000 trabalhos
MAILTO = ""            # opcional: seu e-mail para entrar no "polite pool" do OpenAlex

def buscar_openalex(ror, paginas=PAGINAS, anos="2023-2025"):
    trabalhos, cursor = [], "*"
    for _ in range(paginas):
        params = {"filter": f"authorships.institutions.ror:{ror},publication_year:{anos},type:article",
                  "select": "id,title,publication_year,cited_by_count,authorships", "per-page": 200, "cursor": cursor}
        if MAILTO:
            params["mailto"] = MAILTO
        r = requests.get("https://api.openalex.org/works", params=params, timeout=30)
        r.raise_for_status()
        dados = r.json()
        trabalhos += dados["results"]
        cursor = dados["meta"].get("next_cursor")
        if not cursor:
            break
    return trabalhos

def rede_coautoria(trabalhos, max_autores=MAX_AUTORES):
    ga = nx.Graph()
    descartados = 0
    for w in trabalhos:
        # o mesmo autor pode aparecer duas vezes no mesmo artigo (uma por afiliação); o dict deduplica
        autores = {a["author"]["id"]: a["author"].get("display_name", "?")
                   for a in w.get("authorships", []) if a.get("author", {}).get("id")}
        autores = list(autores.items())
        if len(autores) > max_autores:
            descartados += 1
            continue
        for aid, nome in autores:
            if aid not in ga:
                ga.add_node(aid, nome=nome, artigos=0, citacoes=0)
            ga.nodes[aid]["artigos"] += 1
            ga.nodes[aid]["citacoes"] += w.get("cited_by_count", 0)
        for i in range(len(autores)):
            for j in range(i + 1, len(autores)):
                u, v = autores[i][0], autores[j][0]
                if ga.has_edge(u, v):
                    ga[u][v]["weight"] += 1
                else:
                    ga.add_edge(u, v, weight=1)
    ga.remove_edges_from(nx.selfloop_edges(ga))      # a pegadinha da Seção 4: core_number não aceita laços
    return ga, descartados

def rede_coautoria_sintetica(n_autores=1500, n_artigos=1100, seed=5):
    # produtividade Zipf, artigos assinados por grupos "de laboratório" com vazamento entre grupos
    rng = np.random.default_rng(seed)
    lab = rng.integers(0, 60, n_autores)
    prod = 1 / np.arange(1, n_autores + 1); prod /= prod.sum()
    trabalhos = []
    for i in range(n_artigos):
        n_aut = int(min(40, 1 + rng.zipf(1.8)))
        if rng.random() < 0.01:
            n_aut = int(rng.integers(60, 300))         # mega-colaborações
        primeiro = rng.choice(n_autores, p=prod)
        mesmo_lab = np.where(lab == lab[primeiro])[0]
        pool = np.concatenate([mesmo_lab, rng.choice(n_autores, 5)])
        aut = np.unique(np.concatenate([[primeiro], rng.choice(pool, n_aut - 1)]))
        trabalhos.append({"id": f"W{i}", "cited_by_count": int(rng.zipf(1.6)) - 1,
                          "authorships": [{"author": {"id": f"A{a}", "display_name": f"Autor {a}"}} for a in aut]})
    return trabalhos

try:
    trabalhos = buscar_openalex(ROR_UFRN)
    FONTE_COAUT = "OpenAlex (UFRN, 2023 a 2025)"
except Exception as e:
    print("OpenAlex indisponível:", type(e).__name__, "-> usando rede sintética")
    trabalhos = rede_coautoria_sintetica()
    FONTE_COAUT = "sintética"

ga, descartados = rede_coautoria(trabalhos)
core_a = nx.core_number(ga)
print(f"fonte: {FONTE_COAUT} | trabalhos: {len(trabalhos)} | descartados por terem > {MAX_AUTORES} autores: {descartados}")
print(ga)
print("camadas:", sorted(set(core_a.values())), "| degenerescência:", max(core_a.values()))

fonte: OpenAlex (UFRN, 2023 a 2025) | trabalhos: 1000 | descartados por terem > 25 autores: 102
Graph with 4672 nodes and 26510 edges
camadas: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24] | degenerescência: 24


### 9.1 · Quem está no núcleo?

Um k-core de coautoria com k alto é, quase sempre, **um grupo de pesquisa** (ou dois grupos que publicam juntos). Veja a lista e, se os dados forem reais, procure os nomes no Google Scholar.

In [28]:
kmax_a = max(core_a.values())
nucleo = nx.k_core(ga, kmax_a)
lista = pd.DataFrame([{"autor": ga.nodes[n]["nome"], "artigos": ga.nodes[n]["artigos"], "citações": ga.nodes[n]["citacoes"],
                       "grau": ga.degree(n), "core": core_a[n]} for n in nucleo.nodes()]).sort_values("artigos", ascending=False)
print(f"núcleo principal: {kmax_a}-core, {len(nucleo)} autores, em {nx.number_connected_components(nucleo)} componente(s)")
lista.head(15)

núcleo principal: 24-core, 25 autores, em 1 componente(s)


,autor,artigos,citações,grau,core
19,Rodrigo Torres‐Castro,3,97,38,24
10,Guilherme Fregonezi,2,78,31,24
2,Rocío Martín‐Valero,1,69,24,24
1,Esther Giménez Moolhuyzen,1,69,24,24
4,Jorge Álvarez Rivas,1,69,24,24
5,Luz González-Doniz,1,69,24,24
6,Ana Lista-Paz,1,69,24,24
3,Antonio Tomás Ríos-Cortés,1,69,24,24
0,José Luis Valera,1,69,24,24
8,Jordi Vilaró,1,69,24,24


### 9.2 · Assinaturas em camadas: quatro redes lado a lado

Como a distribuição dos números de núcleo de uma rede real se compara à de modelos clássicos com **o mesmo número de nós e arestas**? Vamos colocar o SocioPatterns, a coautoria e, para cada uma, um Erdős-Rényi e um Barabási-Albert "gêmeos".

> 🧠 **Gestalt em ação · Similaridade (categórica).** Aqui a cor codifica **identidade** (qual rede), não magnitude, então usamos matizes diferentes com ordem fixa. Duas redes reais, dois modelos. O que salta aos olhos é a **cauda**: os modelos aleatórios não produzem um núcleo isolado no fim da escala.

In [29]:
def gemeos(gr, seed=1):
    n, m = gr.number_of_nodes(), gr.number_of_edges()
    er = nx.gnm_random_graph(n, m, seed=seed)
    ba = nx.barabasi_albert_graph(n, max(1, round(m / n)), seed=seed)
    return er, ba

def assinatura(gr, nome):
    c = pd.Series(nx.core_number(gr)).value_counts(normalize=True).sort_index()
    return pd.DataFrame({"k": c.index, "fração de nós": c.values, "rede": nome})

ga_gc = ga.subgraph(max(nx.connected_components(ga), key=len)).copy()     # maior componente da coautoria
er2, ba2 = gemeos(g2); era, baa = gemeos(ga_gc)
assin = pd.concat([assinatura(g2, "SocioPatterns (real)"), assinatura(er2, "ER gêmeo"), assinatura(ba2, "BA gêmeo"),
                   assinatura(ga_gc, f"Coautoria ({FONTE_COAUT})"), assinatura(era, "ER gêmeo"), assinatura(baa, "BA gêmeo")])
assin["família"] = np.where(assin.rede.str.startswith("Socio"), "contatos", np.where(assin.rede.str.startswith("Coaut"), "coautoria", "modelo"))
assin["painel"] = ["SocioPatterns"] * (len(assinatura(g2, "")) + len(assinatura(er2, "")) + len(assinatura(ba2, ""))) + \
                  ["Coautoria"] * (len(assinatura(ga_gc, "")) + len(assinatura(era, "")) + len(assinatura(baa, "")))

fig = px.bar(assin, x="k", y="fração de nós", color="rede", facet_col="painel", barmode="group",
             color_discrete_sequence=[CAT[0], CAT[3], CAT[2], CAT[1], CAT[3], CAT[2]],
             title="Distribuição do número de núcleo: rede real vs modelos com o mesmo n e m")
fig.update_xaxes(matches=None, dtick=1, gridcolor="#e1e0d9")
fig.update_yaxes(gridcolor="#e1e0d9")
fig.update_layout(height=460, plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE,
                  legend=dict(orientation="h", y=-0.22, x=0, title_text=""))
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

> **⏸️ Pare e pense.** O modelo Barabási-Albert tem **uma única shell**: todo nó tem core number igual a $m$. Consegue explicar por quê a partir da regra de crescimento do modelo (cada nó novo chega com exatamente $m$ arestas)? E o Erdős-Rényi, por que concentra tudo em um ou dois valores? A "cebola" com muitas camadas e um núcleo destacado é, portanto, uma **assinatura de redes reais**, não um artefato do algoritmo.

# 10 · Desafios

Seis desafios. Os três primeiros vêm **resolvidos**, para estudo: leia o enunciado, tente antes de abrir a solução, depois compare. Os três últimos **não têm solução no notebook**. Eles têm critérios de aceitação e pistas, e alguns tocam em perguntas de pesquisa em aberto. Entregue-os como um notebook separado.

| # | Desafio | Estado | Dificuldade |
|---|---|---|---|
| 1 | k-shell na mão e reconstrução do core number | resolvido | ★☆☆ |
| 2 | Hubs sem núcleo: quem são e por quê | resolvido | ★★☆ |
| 3 | Ataque à rede: grau vs núcleo vs aleatório | resolvido | ★★☆ |
| 4 | O núcleo ao longo do dia (k-core temporal) | aberto | ★★★ |
| 5 | s-core: núcleos ponderados | aberto | ★★★ |
| 6 | Mega-colaborações, k-truss e a hipótese das citações | aberto | ★★★★ |

## Desafio 1 · k-shell na mão (resolvido)

**Enunciado.** Sem usar `nx.k_shell` nem `nx.core_number`, apenas `nx.k_core`:

(a) escreva `k_shell_na_mao(G, k)` que retorne o conjunto de nós da k-shell;
(b) escreva `core_number_na_mao(G)` que reconstrua o dicionário completo de core numbers;
(c) prove (com `assert`) que ambas coincidem com o NetworkX no grafo dos slides e no SocioPatterns;
(d) discuta a complexidade da sua solução em relação ao algoritmo da Seção 6.

<details><summary><b>Pista</b></summary>
A relação $k\text{-shell} = k\text{-core} \setminus (k+1)\text{-core}$ é tudo que você precisa para (a). Para (b), pense em quantas vezes um nó "sobrevive".
</details>

In [30]:
# ---- Solução do Desafio 1 ----
def k_shell_na_mao(G, k):
    return set(nx.k_core(G, k).nodes()) - set(nx.k_core(G, k + 1).nodes())

def core_number_na_mao(G):
    # um nó tem core number k se sobrevive ao k-core mas não ao (k+1)-core
    cn = {n: 0 for n in G.nodes()}
    k = 1
    while True:
        sobreviventes = nx.k_core(G, k).nodes()
        if len(sobreviventes) == 0:
            break
        for n in sobreviventes:
            cn[n] = k
        k += 1
    return cn

for G in [g, g2]:
    for k in set(nx.core_number(G).values()):
        assert k_shell_na_mao(G, k) == set(nx.k_shell(G, k).nodes())
    assert core_number_na_mao(G) == nx.core_number(G)
print("✅ (a), (b) e (c) verificados no grafo dos slides e no SocioPatterns")

✅ (a), (b) e (c) verificados no grafo dos slides e no SocioPatterns


**(d) Complexidade.** `core_number_na_mao` chama `nx.k_core` uma vez por camada, e cada chamada custa $O(n+m)$ (o NetworkX calcula `core_number` internamente e depois filtra). Com $K$ camadas, o custo é $O(K \cdot (n+m))$. É sempre pior que o algoritmo de Batagelj e Zaversnik, que faz tudo em uma única passada $O(n+m)$. No SocioPatterns, com $K = 17$, são 17 passadas onde bastaria uma. A lição é: **`core_number` é a primitiva, tudo o mais é filtro sobre ela**.

## Desafio 2 · Hubs sem núcleo (resolvido)

**Enunciado.** Na Seção 7.5 marcamos em vermelho visitantes com grau no top 10% e core number no máximo metade da degenerescência.

(a) Para cada hub sem núcleo, calcule a **fração dos vizinhos que estão na casca** (core ≤ 4) e compare com a mesma fração para os nós do núcleo principal;
(b) calcule o **coeficiente de agrupamento local** dos dois grupos e compare;
(c) desenhe a **vizinhança (ego-network)** do hub sem núcleo de maior grau usando figura e fundo (ego em vermelho, vizinhos coloridos pelo core number, o resto apagado);
(d) escreva, em uma frase, por que grau alto não bastou para colocá-lo no núcleo.

In [31]:
# ---- Solução do Desafio 2 ----
hubs_sem_nucleo = tab2[tab2.tipo == "hub sem núcleo"].visitante.tolist()
nucleo_ids = [n for n in nos2 if core2[n] == kmax]
clust = nx.clustering(g2)

def fracao_vizinhos_casca(n, limite=4):
    viz = list(g2.neighbors(n))
    return np.mean([core2[v] <= limite for v in viz]) if viz else np.nan

comparacao = pd.DataFrame({
    "grupo": ["hubs sem núcleo"] * len(hubs_sem_nucleo) + [f"núcleo ({kmax}-core)"] * len(nucleo_ids),
    "visitante": hubs_sem_nucleo + nucleo_ids})
comparacao["fração de vizinhos na casca"] = comparacao.visitante.map(fracao_vizinhos_casca)
comparacao["agrupamento local"] = comparacao.visitante.map(clust)
comparacao["grau"] = comparacao.visitante.map(grau2)
comparacao.groupby("grupo")[["grau", "fração de vizinhos na casca", "agrupamento local"]].agg(["median", "mean"]).round(3)

grau         fração de vizinhos na casca         \
                 median    mean                      median   mean   
grupo                                                                
hubs sem núcleo    26.0  26.000                       0.038  0.038   
núcleo (17-core)   25.0  26.219                       0.000  0.009   

                 agrupamento local         
                            median   mean  
grupo                                      
hubs sem núcleo              0.206  0.206  
núcleo (17-core)             0.637  0.605

In [32]:
# (c) ego-network do hub sem núcleo de maior grau
if hubs_sem_nucleo:
    ego = max(hubs_sem_nucleo, key=lambda n: grau2[n])
    vizinhos = set(g2.neighbors(ego))
    cores_ego = [DESTAQUE if n == ego else (rampa_azul([core2[n]], 1, kmax)[0] if n in vizinhos else CINZA_NO) for n in nos2]
    tam_ego = [22 if n == ego else (12 if n in vizinhos else 6) for n in nos2]
    opac = [1.0 if (n == ego or n in vizinhos) else 0.25 for n in nos2]
    h_ego = g2.subgraph(vizinhos | {ego})
    fig = figura_rede([traco_arestas(g2, pos_spring2, largura=0.3, opacidade=0.15),
                       traco_arestas(h_ego, pos_spring2, cor=AZUIS[3], largura=1.2, opacidade=0.8),
                       traco_nos(g2, pos_spring2, cores_ego, tam_ego, hover2, rotulos=False, opacidade=opac)],
                      f"Ego-network do visitante {ego} · grau {grau2[ego]}, core {core2[ego]} · vizinhos coloridos pelo core", altura=600)
    fig.show()
    print(f"vizinhos de {ego} por core number:", dict(sorted(pd.Series([core2[v] for v in vizinhos]).value_counts().items())))
    print("arestas entre os vizinhos (triângulos fechados):", h_ego.number_of_edges() - len(vizinhos))

vizinhos de 42 por core number: {4: 1, 5: 2, 6: 2, 7: 2, 8: 15, 12: 4}
arestas entre os vizinhos (triângulos fechados): 67


**(d)** Um hub sem núcleo encontrou muita gente, mas seus contatos pertencem majoritariamente à casca e **quase não se conhecem entre si** (agrupamento baixo). Quando a casca é removida, ele perde a maioria dos vizinhos de uma vez e cai junto. O core number mede **vizinhança mútua**, não volume de contatos.

## Desafio 3 · Ataque à rede (resolvido)

**Enunciado.** Imagine que você pode vacinar (remover) visitantes da exposição, um a um, para fragmentar a rede de contatos. Compare três estratégias:

* **por grau**: remove sempre o nó de maior grau *na rede atual* (recalcula a cada passo);
* **por núcleo**: remove o nó de maior core number *na rede atual* (recalcula a cada passo; desempate por grau);
* **aleatório**: remove um nó ao acaso (média de 10 execuções).

Meça, após cada remoção, o **tamanho do maior componente** dividido pelo n original. Plote as três curvas até remover 30% dos nós e responda: qual estratégia fragmenta mais rápido? A resposta muda entre o começo e o fim?

> 🧠 **Gestalt em ação · Continuidade + Figura e fundo.** Três linhas categóricas, e a área abaixo da curva vencedora preenchida com um cinza leve: a área é a "dose total de contágio evitada".

In [33]:
# ---- Solução do Desafio 3 ----
def ataque(G, estrategia, fracao=0.30, seed=0):
    h = G.copy(); n0 = h.number_of_nodes(); rng = random.Random(seed)
    curva = [1.0]
    for _ in range(int(fracao * n0)):
        if estrategia == "grau":
            alvo = max(h.degree, key=lambda x: x[1])[0]
        elif estrategia == "núcleo":
            cn, dg = nx.core_number(h), dict(h.degree())
            alvo = max(h.nodes(), key=lambda v: (cn[v], dg[v]))
        else:
            alvo = rng.choice(list(h.nodes()))
        h.remove_node(alvo)
        maior = max((len(c) for c in nx.connected_components(h)), default=0)
        curva.append(maior / n0)
    return np.array(curva)

curvas = {"por grau": ataque(g2, "grau"), "por núcleo": ataque(g2, "núcleo"),
          "aleatório (média de 10)": np.mean([ataque(g2, "aleatório", seed=s) for s in range(10)], axis=0)}
x = np.arange(len(curvas["por grau"])) / g2.number_of_nodes()

vencedora = min(curvas, key=lambda k: curvas[k].sum())
fig = go.Figure()
for (nome, y), cor in zip(curvas.items(), CAT):
    fig.add_scatter(x=x, y=y, mode="lines", name=nome, line=dict(color=cor, width=2.5),
                    fill="tozeroy" if nome == vencedora else None, fillcolor="rgba(11,11,11,0.06)",
                    hovertemplate=nome + "<br>removidos: %{x:.1%}<br>maior componente: %{y:.1%}<extra></extra>")
fig.update_layout(title="Fragmentação da rede de contatos sob três estratégias de remoção", xaxis_title="fração de nós removidos",
                  yaxis_title="tamanho relativo do maior componente", height=460, plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE,
                  legend=dict(orientation="h", y=1.08, x=0), xaxis=dict(tickformat=".0%", gridcolor="#e1e0d9"),
                  yaxis=dict(tickformat=".0%", gridcolor="#e1e0d9"), hovermode="x unified")
fig.show()
print("área sob a curva (menor = fragmenta mais):", {k: round(float(v.mean()), 3) for k, v in curvas.items()})

área sob a curva (menor = fragmenta mais): {'por grau': 0.833, 'por núcleo': 0.838, 'aleatório (média de 10)': 0.846}


**Discussão.** A remoção por grau costuma vencer no **início**, porque hubs sem núcleo derrubam muitas arestas de uma vez. Remover pelo núcleo ataca a **reserva de coesão** da rede e tende a ganhar quando o objetivo é impedir que um surto se sustente (o núcleo máximo é o lugar onde uma epidemia SIS sobrevive por mais tempo, resultado de Kitsak et al., 2010). Duas medidas, duas perguntas diferentes: *fragmentar* não é o mesmo que *conter a propagação*. Guarde essa tensão para o Desafio 4.

## Desafio 4 · O núcleo ao longo do dia (aberto)

A rede que analisamos é **estática**: agregamos todos os contatos do dia. Mas o arquivo tem `timestamp`. O 17-core existiu o dia inteiro, ou foi um grupo que chegou junto e ficou junto por uma hora?

**Tarefa.** Divida o dia em janelas de tempo (por exemplo, 30 minutos, com ou sem sobreposição), construa um grafo por janela e calcule `core_number` em cada uma. Depois:

1. Plote a **degenerescência** ao longo do tempo (uma série temporal) e o tamanho do núcleo principal de cada janela.
2. Construa uma **animação** (como a da Seção 3) mostrando, no *layout* fixo da rede agregada, quais nós estão no núcleo principal em cada janela. Nós que entram no núcleo devem "acender"; os que saem, "apagar" (destino comum).
3. Responda: os membros do 17-core agregado formam núcleo em **alguma** janela individual? Qual é o k máximo que eles alcançam dentro de uma única janela?
4. Proponha e implemente uma definição de **núcleo temporal** (por exemplo, "nó que está no k-core de pelo menos $\tau$ janelas consecutivas") e compare com o núcleo estático.

**Critérios de aceitação.** A animação deve ter *slider* e botão de tocar; a série temporal deve ter *hover* com o horário legível (não o timestamp bruto); a resposta ao item 3 deve vir com números; a definição do item 4 deve ser justificada em um parágrafo.

**Pista.** `pd.to_datetime(df.timestamp, unit="s")` e `pd.Grouper(freq="30min")` resolvem a janela. Cuidado com janelas em que o grafo fica vazio.

**Por que é aberto.** Não há consenso na literatura sobre a "melhor" definição de k-core em redes temporais (existem span-cores, (k, h)-cores, cores em janelas deslizantes). A sua definição pode ser tão boa quanto qualquer outra, desde que seja **defendida**.

In [ ]:
# ---- Desafio 4: seu código aqui ----

## Desafio 5 · s-core: núcleos ponderados (aberto)

O k-core ignora os pesos. Mas na exposição, encontrar alguém 40 vezes é muito diferente de encontrar uma vez. Eidsaa e Almaas (2013) propuseram o **s-core**: em vez de exigir "pelo menos k vizinhos", exige-se "**força** (soma dos pesos das arestas) pelo menos s" dentro do subgrafo. O algoritmo de descascar funciona igual, removendo repetidamente o nó de menor força.

**Tarefa.**

1. Implemente `s_core_number(G, weight="weight")` no estilo da função `descascar` da Seção 3. Como a força é contínua, o "número" de cada nó é a **força no momento da remoção** (monotônica, como o k).
2. Verifique que, com todos os pesos iguais a 1, sua função devolve exatamente `nx.core_number`.
3. Aplique ao SocioPatterns e compare os rankings: quantos nós do 17-core não ponderado estão no s-core máximo? Há nós que **sobem** de camada com os pesos? Quem são?
4. Desenhe as duas cebolas lado a lado com o **mesmo layout**, para que o olho compare camada a camada.
5. (Ousado) Adapte o algoritmo de Batagelj e Zaversnik para força inteira. Onde a ideia dos baldes quebra quando o peso é real?

**Critérios de aceitação.** Teste do item 2 com `assert`; figura do item 4 com hover mostrando core e s-core do mesmo nó; discussão do item 5 mesmo que a implementação não fique pronta.

**Por que é aberto.** O NetworkX não implementa s-core. Existem pacotes externos, mas a graça é que a sua implementação pode ser a referência da turma.

In [ ]:
# ---- Desafio 5: seu código aqui ----

## Desafio 6 · Mega-colaborações, k-truss e a hipótese das citações (aberto)

Na Seção 9 descartamos artigos com mais de 25 autores. Foi uma escolha arbitrária, e ela decidiu quem é o "núcleo da UFRN".

**Tarefa.**

1. Refaça a rede de coautoria com cortes de 10, 25, 50, 100 e sem corte. Para cada, registre degenerescência, tamanho do núcleo principal e **quais autores** estão nele. Mostre em uma figura como o núcleo "vira" quando o corte muda (uma matriz autor × corte, com fechamento visual dos blocos, é uma boa opção).
2. Compare o k-core com o **k-truss** na mesma rede. Qual dos dois é mais robusto à presença de um artigo com 200 autores? Explique com base nas definições.
3. Teste a hipótese **"autores com core number maior recebem mais citações por artigo"**. Use `citacoes / artigos` de cada autor. Mostre a distribuição por camada (violinos como na Seção 8.2) e reporte uma correlação com intervalo de confiança por *bootstrap*.
4. Se você usou os dados reais, escolha **um** autor do núcleo e escreva três linhas sobre o grupo de pesquisa ao qual ele pertence, a partir do que a rede mostra (não do Lattes).

**Critérios de aceitação.** Item 1 com pelo menos uma figura interativa; item 2 com um exemplo numérico; item 3 com a correlação, o IC e um parágrafo sobre **por que correlação aqui não é causalidade** (cite ao menos dois confundidores plausíveis).

**Por que é aberto.** A relação entre posição no núcleo de coautoria e impacto científico é um tema de pesquisa ativo em cientometria, e depende de área, período e de como se trata a hiperautoria. Não existe resposta única; existe resposta bem argumentada.

In [ ]:
# ---- Desafio 6: seu código aqui ----

# 11 · Quiz relâmpago e checklist de saída

Responda de cabeça e só depois abra a resposta.

<details><summary><b>1. Um nó com grau 50 pode ter core number 1?</b></summary>
Sim. Uma estrela com 50 pontas: o centro tem grau 50 e core number 1, porque todas as pontas têm grau 1 e caem na primeira rodada, levando o centro junto.
</details>

<details><summary><b>2. O 3-core de um grafo pode ser desconexo?</b></summary>
Sim. Dois cliques K4 separados formam, juntos, um 3-core com dois componentes. Núcleo não implica conectividade.
</details>

<details><summary><b>3. Adicionar uma aresta pode diminuir o core number de algum nó?</b></summary>
Nunca. Core number é monotônico em relação a subgrafos: $H \subseteq G \Rightarrow c_H(v) \leq c_G(v)$.
</details>

<details><summary><b>4. Por que a rede SocioPatterns não tem 14-, 15- nem 16-shell?</b></summary>
Porque ao remover a 13-shell o que resta já satisfaz grau ≥ 17 para todos os nós. Camadas intermediárias só existem se houver nós que "sobrevivem" com grau exatamente 14, 15 ou 16, e não há.
</details>

<details><summary><b>5. k-truss e k-core: qual é sempre subconjunto de qual?</b></summary>
O k-truss está sempre contido no (k−1)-core. Se cada aresta de um nó está em k−2 triângulos, o nó tem pelo menos k−1 vizinhos no subgrafo.
</details>

### ✅ Checklist de saída

- [ ] Sei escrever a definição de k-core, core number e k-shell sem consultar.
- [ ] Consigo descascar um grafo de 10 nós no papel e acertar o resultado de `nx.core_number`.
- [ ] Entendo por que `nx.core_number` é $O(n+m)$ e por que a versão ingênua não é.
- [ ] Sei a diferença entre `k_core`, `k_shell`, `k_crust`, `k_corona`, `onion_layers` e `k_truss`.
- [ ] Consigo explicar, com um exemplo real, por que hub ≠ núcleo.
- [ ] Consigo desenhar a "cebola" de uma rede e ler a assinatura em camadas.

### 📚 Referências

* Seidman, S. B. (1983). Network structure and minimum degree. *Social Networks*, 5(3).
* Batagelj, V. e Zaversnik, M. (2003). An O(m) algorithm for cores decomposition of networks. arXiv:cs/0310049.
* Kitsak, M. et al. (2010). Identification of influential spreaders in complex networks. *Nature Physics*, 6.
* Hébert-Dufresne, L., Grochow, J. A. e Allard, A. (2016). Multi-scale structure and topological anomaly detection via a new network statistic: the onion decomposition. *Scientific Reports*, 6.
* Eidsaa, M. e Almaas, E. (2013). s-core network decomposition: a generalization of k-core analysis to weighted networks. *Physical Review E*, 88.
* Isella, L. et al. (2011). What's in a crowd? Analysis of face-to-face behavioral networks. *J. Theor. Biol.*, 271. (SocioPatterns INFECTIOUS)
* Documentação: [networkx.algorithms.core](https://networkx.org/documentation/stable/reference/algorithms/core.html) · [OpenAlex API](https://docs.openalex.org)